# Assignment - Visualization

Make sure to **read the instructions** completely. Failing to follow the instructions will have a negative impact on your grade!

In this assignment, we will explore ways to visualize the data, starting with simple plots for early exploration and build towards more interactive methods. As we want to go beyond static plots possible with for example matplotlib or seaborn, we will use the [Bokeh](https://www.bokeh.org/) library. This means the exercises **must** be implemented with Bokeh to receive points.

This assignment includes four parts (Assignmets V.1 to V.4) which provide different amounts of points. V.1 to V.3 are independent assignments, but V.4 builds on the previous assignments.
 
**Before you submit**, make sure that you are not accidentaly using any global variables. Restart the kernel (wiping all global variables) and run the code from top to bottom by clicking "Kernel" => "Restart & Run all" in the menu bar at the top.

### API documentation

note: you can find the used versions by checking the `environment.yml` file or `running conda list -n dsait4000vis` from the command line

* [Bokeh](https://docs.bokeh.org/) (also has examples)
* [Pandas](https://pandas.pydata.org/docs/)
* [NumPy](https://numpy.org/doc/stable/)
* [SciPy](https://docs.scipy.org/doc/scipy/)


### Rules
For this assignment the following rules apply:

**General**
 * The assignment should be completed in the **groups** you have already assembled for the previous assignment. Work together on all parts and do not distribute the work. Everyone is responsible for everything. That also means if there is fraud in a part it will be considered fraud by the whole group and voids the complete assignment, even if you distributed the parts of the assignment.
 * Any kind of intergroup discussion will be considered fraud and both the parties will be punished.
 * All code must be written within group. All external help, with the exception of Python/library documentation and the lecture slides, will be considered fraud.
 * Do not add any libarires to those already included notebook (i.e., `numpy`, `scipy`, `pandas`, and `bokeh`)
 * Add functionality from bokeh, like ```from bokeh.io import output_notebook```, on top of the block where you need it.
 * **Do not rename the notebook file**

**AI Use**
 * The use of AI for this assignment is permitted. If you use AI, please explain what AI was used for and how you made sure you used it responsibly in the section at the end of this file. 
 * Every group member should be able to explain all facets of the entire assignment. 

**Grading**
 * Exercise may (partially) be graded automatically, using additional hidden tests. These tests will check your solution for different (unseen) inputs.
 * If these tests pass without error (warnings are allowed) then you receive full points.
 * Do not make any assumptions on the input data (such as number or type of columns) unless specified otherwise. Doing so may result in the tests failing and thus 0 points.
 * Your grade is calculated as the sum of the achieved points and will be rounded to the closest 0.1 point.
 
**Submissions**
 * Submit your code to Brightspace
 * For this assignment you must *only* submit the notebook (`*.ipynb`) file filled in with your solution.
 * Refer to Brightspace for the exact deadline.
 * We will consider submissions that are handed in up to 3 hours after the deadline but with a penalty as follows:
 * For every completed 20 minutes past the deadline the grade will be reduced by 0.5 (e.g., with the deadline at 12:00, a submission at 12:19 will get the full grade, a submission at 12:20 will receive a 0.5 grade penalty).

In [ ]:
# We do the basic setup here.
# Do not (!) change this cell! All changes will be reset during grading!

# numpy and pandas
import numpy as np
import pandas as pd

# We import a range of bokeh functionality that will likely be needed already
# if you need more, import it at the top of the corresponding cell
from bokeh.plotting import figure, gridplot, show
from bokeh.layouts import column
from bokeh.transform import factor_cmap, linear_cmap
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter, BasicTicker, PrintfTickFormatter, CustomJS, Select, axes
from bokeh.models.annotations import Label
from bokeh.io import output_notebook
output_notebook() # activate Bokeh output to Jupyter notebook

# You can use this to calculate the density plot in Assignment V.3
from scipy.stats import gaussian_kde

# We load the adult data  (https://archive.ics.uci.edu/dataset/2/adult) as in previous assignments (note, we keep the headers, to use them in the pandas dataframe)
# Note: we do not change ? values to nan, but rather keep them as ? so they will be shown in the visualizations
data = pd.read_csv("datasets/Visualization/adult_all.csv")

data.head()

### Assignment V.1 (5 points)

Getting an overview of the data distribution

In this first assignment, we want to use visualization to get a quick overview of the distribution of the different attributes/columns. This can often be helpful to make sure that we use models that match the distribution downstream or to quickly identify for example whether the data is distributed over all classes equally if the attribute is intended to be used for training a model.

A good way to quickly plot the distribution is a simple bar chart for each attribute. However, this will only allow us to look at one attribute at a time. In class we have discussed small multiples. We can use this concept to plot all attributes in a grid of plots. 

For this assignment do the following:
* Create a grid of bar charts, with one chart per column in the dataset. The attribute values shall be on the x-axis and the count of each value on the y-axis. Look at pandas `value_counts` function to get the latter.
  Hint: The `figure()` function works slightly different for numerical and categorical data.
* Make sure that the charts and axes are well labeled (for example like in the screenshot below). It will be hard to properly label the x-axis with many items (e.g., native-country). Do this as well as you can, you will likely not find a perfect solution for all attributes.
* Add a tooltip to show the values on hover (see the education chart, top-right, in the image below).

The result should look similar to the below.

[<img src="assets/barcharts.png" alt="Small Multiples showing the distributions of all columns in the dataset." width="500"/>](assets/barcharts.png)

In [ ]:
def make_bar_chart(col):
    series = data[col]
    is_numeric = pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series)

    if is_numeric:
        counts = series.value_counts().sort_index()
        values = counts.index.to_numpy()
        steps = np.diff(values)
        width = max(0.9 * steps.min(), (values.max() - values.min()) / 500) if len(steps) else 1.0
        source = ColumnDataSource(dict(value=values, count=counts.values))
        p = figure(frame_width=290, frame_height=170, title=col, toolbar_location=None, tools="")
        p.xaxis.formatter = NumeralTickFormatter(format="0.[0]a")
        p.xaxis.ticker = BasicTicker(desired_num_ticks=4)
    else:
        labels = series.dropna().astype(str).str.strip().str.replace(r"\.$", "", regex=True)
        counts = labels.value_counts()
        values = list(counts.index)
        width = 0.8
        source = ColumnDataSource(dict(value=values, count=counts.values))
        p = figure(frame_width=290, frame_height=170, title=col, x_range=values, toolbar_location=None, tools="")
        p.xaxis.major_label_orientation = np.pi / 2
        p.xaxis.major_label_text_font_size = "7pt" if len(values) <= 20 else "5pt"

    p.vbar(x="value", top="count", width=width, source=source, fill_color="steelblue", line_color=None,
           hover_fill_color="orange")
    p.add_tools(HoverTool(tooltips=[(col, "@value"), ("count", "@count{0,0}")]))
    p.xaxis.axis_label = col
    p.yaxis.axis_label = "count"
    p.yaxis.formatter = NumeralTickFormatter(format="0,0")
    p.y_range.start = 0
    return p

show(gridplot([make_bar_chart(col) for col in data.columns], ncols=4))


#### A bar chart like this doesn't serve every attribute equally well

Look closely at the grid you just built. For some attributes, a few attribute values accounts for the overwhelming majority of the data, resulting in a few very long bars. The remaining attribute values  Such a bar chart does not allow us to see the distribution of the data in a meaningful way. 

* Identify the numerical attributes where this happens. For this task, you can select attributes by name. 
* For just those attributes, rebuild the chart as a **binned histogram**. Do not make any changes to the other bar charts. You may need to try a few different bin sizes to find a suitable one. **Justify your choice for the bin sizes in 50 words or less** in the response field below. 
* Choose an appropriate scale for the Y axis that shows the data distribution properly. **Justify your choice for the Y axis scale in 50 words or less** in the response field below.  

In [ ]:
skewed_cols = [c for c in ["capital-gain", "capital-loss"]
               if c in data.columns and pd.api.types.is_numeric_dtype(data[c])]
bin_widths = {"capital-gain": 2500, "capital-loss": 100}

def make_histogram(col):
    values = data[col].dropna()
    width = bin_widths.get(col, (values.max() - values.min()) / 40 or 1)
    start = np.floor(values.min() / width) * width
    n_bins = max(1, int(np.ceil((values.max() - start) / width)))
    counts, edges = np.histogram(values, bins=start + width * np.arange(n_bins + 1))
    source = ColumnDataSource(dict(left=edges[:-1], right=edges[1:], count=counts,
                                   top=np.where(counts > 0, counts, 0.5)))
    p = figure(frame_width=290, frame_height=170, title=col, y_axis_type="log", toolbar_location=None, tools="")
    p.quad(left="left", right="right", top="top", bottom=0.5, source=source,
           fill_color="steelblue", line_color="white", hover_fill_color="orange")
    p.add_tools(HoverTool(tooltips=[(col, "@left{0,0.[##]} to @right{0,0.[##]}"), ("count", "@count{0,0}")]))
    p.xaxis.axis_label = col
    p.yaxis.axis_label = "count (log scale)"
    p.xaxis.formatter = NumeralTickFormatter(format="0.[0]a")
    p.xaxis.ticker = BasicTicker(desired_num_ticks=4)
    p.y_range.start = 0.5
    return p

show(gridplot([make_histogram(col) if col in skewed_cols else make_bar_chart(col) for col in data.columns], ncols=4))


#### Response: Justification for binning strategy and Y axis scale 

**Bins:** Widths of 2,500 (capital-gain) and 100 (capital-loss) give about 40 round-number bins. Wider bins hide the shape of the small non-zero groups; narrower ones leave many bins with one or two points.

**Y axis:** Logarithmic. 92-95% of rows are 0, so on a linear axis the zero bin dwarfs everything else and all other bars are invisible. A log scale shows every magnitude.


#### Notes on design decisions (V.1)

* **Missing values:** empty values (NaN) are not drawn; `?` is kept as its own bar, as the setup cell asks.
* **Labels:** values differing only by a trailing `.` are merged for display (`<=50K.` and `<=50K`); the data itself is unchanged.
* **Histograms:** only `capital-gain` and `capital-loss` (92% / 95% zeros). `hours-per-week` stays a bar chart (its spike at 40 still leaves the other bars readable); `fnlwgt` stays a bar chart with a minimum bar width so its thin bars stay visible.
* **Log axis:** bars start at 0.5 so bins with a single point stay visible; the tooltip shows the true count.
* **Layout:** equal plot sizes, compact x tick labels (e.g. 500k), a smaller font for `native-country`'s 42 labels.


### Assignment V.2 (5 points)

Visualizing possible correlations and some interactivity

In this assignment, we want to visualize the correlation between attributes. To keep this simple we only use the numerical variables here. You can use pandas `corr()` [function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) directly on the dataframe using the `numeric_only = True` argument. The output will contain only the results from using numerical columns. It should be noted though, that there are also ways to calculate dependencies or correlation between catagorical (e.g., [Chi-Square_test](https://en.wikipedia.org/wiki/Chi-square_test), [Cramér's V](https://en.wikipedia.org/wiki/Cramér%27s_V)) or categorical and numerical (e.g., [One-way ANOVA test](https://en.wikipedia.org/wiki/F-test#One-way_ANOVA_example)) values.

The output will be a matrix of pairwise values. It is common to visualize such data in a heatmap (images below).

* Calculate the correlation of the numerical columns using pandas `corr()`, using Pearson correlation (this is the default of the function).
* Create a heatmap showing the values. The data format output by `corr()` might not be very useful. Look for example at [pandas `stack()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html) for possible ways to transforma the data into a typical dataframe.
* A heatmap uses x and y position for the attributes and color for the value, choose a fitting colormap and **justify it below in 50 words or less**. (Note: the color chosen in the example below might or might not be good. Do **not** use the example to justify your choice.)
* Make sure the chart is legible by adding axes, legends, a color bar, or similar. It is up to you to decide what is needed.

We now want to extend our chart to be able to switch between the Pearson and Spearman correlation interactively using a dropdown menu directly in the chart.
* Add a `Select` dropdown with the two options.
* Calculate both correlation values and add a plot for each to your figure.
* Use `js_on_change` on the added Select to add code to switch between the two plots.

[<img src="assets/heatmap.png" alt="Heatmap showing correlations" width="500"/>](assets/heatmap.png)
[<img src="assets/heatmap_select.png" alt="Heatmap showing correlations with open selection widget" width="500"/>](assets/heatmap_select.png)


In [ ]:
# We calculate both matrices
correlation_matrices = {
    "pearson": data.corr(numeric_only = True, method="pearson"),
    "spearman": data.corr(numeric_only = True, method="spearman")
}

# this is the default correlation method
correlation_method = "spearman"

# YOUR CODE HERE
from bokeh.palettes import RdBu11
from bokeh.models import ColorBar

methods = ["Spearman", "Pearson"]
corr_data_pearson = correlation_matrices["pearson"].stack().reset_index().rename(columns={0: "corr_values"})
corr_data_spearman = correlation_matrices["spearman"].stack().reset_index().rename(columns={0: "corr_values"})

cmap = linear_cmap(field_name="corr_values", palette=RdBu11, low=-1, high=1)
attributes = pd.unique(corr_data_pearson['level_0'])

p = figure(title='Correlation', x_range=attributes, y_range=attributes, toolbar_location="above")

spearman_p = p.rect(x="level_1", y="level_0", width=1, height=1, fill_color=cmap, source=ColumnDataSource(corr_data_spearman), line_color="black")
pearson_p = p.rect(x="level_1", y="level_0", width=1, height=1, fill_color=cmap, source=ColumnDataSource(corr_data_pearson), line_color="black", visible=False)

dropdown = Select(title="Correlation Method", value=methods[0], options=methods)
dropdown.js_on_change("value", CustomJS(args=dict(spearman=spearman_p, pearson=pearson_p), 
        code="""
            spearman.visible = (this.value == "Spearman");
            pearson.visible = (this.value == "Pearson");    
        """))

p.add_layout(ColorBar(color_mapper=cmap["transform"]), "right")

layout = column(dropdown, p)
show(layout)


#### Response: Justification for colormap

Pearson and Spearman correlations always range from -1 to 1 indicating the correlation between values. Blue and red are adequate colors for this color map because they are color blind-friendly and will allow all audiences to visualize the gradient of differences in the data without the positive or negative biases associated with green and red. Because this gradient passes through a white color zone near the zero values, the grid lines are black for easier distinction. 

### Assignment V.3 (5 points)

In this assignment, we will visualize the relationship between two numerical columns and one categorical column from the dataset. We already extract the names of the first two numerical columns below; the easiest is to just use those, but instead of fixing the categorical column we now want it to be **user-selectable**, similar to Assignment V.2. (Remember that we might test this with a different dataset, so do not pick columns by name.)

The goal here is to visualize the complete data of the selected columns. We will proceed in two steps again.

In the first step, create a scatterplot to visualize the selected columns.
* A scatterplot is a common representation, bokeh provides it using `scatter`.
* Use the x and y axes for the numerical values.
* Add a `Select` dropdown listing the categorical columns in the dataset, and use it to choose which categorical column is encoded in the plot (e.g. through color). Use `js_on_change` to update the plot when a different column is selected, similar to Assignment V.2.
* Choose a proper encoding for the categorical column and **justify** it in the field below the code **in 50 words or less**. (Note: this encoding was omitted in the example figure below.)
* Make sure the chart is legible by adjusting mark sizes, adding axes, legends, a color bar, or similar, etc. It is up to you to decide what is needed.

Even with putting in a lot of effort, you will notice that there will be a lot of overplotting, even with 'just' ~50k points in the dataset. In the example below, you see iso-density lines added to show how the points distribute. We have already added the `kde` function to [estimate the density.](https://en.wikipedia.org/wiki/Multivariate_kernel_density_estimation) Use this to calculate the point density. This is taken from an [example in the Bokeh documenation](https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html). Look at the [example](https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html) and adjust it to your needs.
* Add the density information to your plot. (This only needs to depend on the two numerical columns, not on the selected categorical column.)

[<img src="assets/scatterplot.png" alt="Scatterplot with density indicator" width="700"/>](assets/scatterplot.png)


In [ ]:
# kde calculation taken from https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html
# look at the example on how it is used. You can use the plot code also for inspiration, but make sure to adjust it for proper results
# note, N is the kernel size, larger values will make this lengthy to calculate but give more smooth results.
# When you use it consider using small values for N during testing.
def kde(x, y, N):

    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    X, Y = np.mgrid[xmin:xmax:N*1j, ymin:ymax:N*1j]
    positions = np.vstack([X.ravel(), Y.ravel()])
    values = np.vstack([x, y])
    kernel = gaussian_kde(values)
    Z = np.reshape(kernel(positions).T, X.shape)

    return X, Y, Z

# we extract the names of the first two numerical columns
num_cols = []
num_cols.append(data.select_dtypes(include="number").columns[0])
num_cols.append(data.select_dtypes(include="number").columns[1])
# and all categorical columns, to populate the dropdown.
cat_cols = list(data.select_dtypes(include=["object", "category"]).columns)
# You can just use the above columns to set up your visualization. Feel free to change these for testing.
# For the submission, you can safely assume that at least 2 numerical and 1 categorical columns are available in the dataset.

# YOUR CODE HERE
from bokeh.palettes import Category10, Category20, turbo

def cat_as_str(col):
    "Categorical column as strings. Ensures proper handling of NaN values"
    return data[col].astype(str).mask(data[col].isna(), "NaN")

# one source: the two numerical columns plus one (string) column per categorical column
source = ColumnDataSource({"x": data[num_cols[0]], "y": data[num_cols[1]]}
                          | {c: cat_as_str(c) for c in cat_cols})

# density of the two numerical columns
# ensures that NaN and inf values are handled properly before KDE
x = data[num_cols[0]].to_numpy(dtype=float, na_value=np.nan)
y = data[num_cols[1]].to_numpy(dtype=float, na_value=np.nan)
finite = np.isfinite(x) & np.isfinite(y)
X, Y, Z = kde(x[finite], y[finite], N=100)
levels = np.linspace(Z.min(), Z.max(), 10)[1:]

def make_scatterplot(cat_col):
    col_val_range = sorted(pd.unique(cat_as_str(cat_col)))

    if len(col_val_range) <= 10:
        # At most 10 unique values
        palette = Category10[10]
    elif len(col_val_range) <= 20:
        # At most 20 unique values
        palette = Category20[20]
    else:
        # More than 20 unique values
        # Get however many colors need for each unique val in the column (at most 256)
        palette = turbo(min(len(col_val_range), 256))

    p = figure(width=1000, height=750, x_axis_label=num_cols[0], y_axis_label=num_cols[1])
    # Proper formatting of the y-axis to show numbers in a more readable format
    p.yaxis.formatter = NumeralTickFormatter(format="0.[0]a")
    # The scatter plot
    p.scatter("x", "y", source=source, size=4, alpha=0.3,
              color=factor_cmap(cat_col, palette, col_val_range), legend_group=cat_col)
    # The density contour
    p.contour(X, Y, Z, levels, line_color="black", line_alpha=0.5)
    
    p.add_layout(p.legend[0], "right")
    # keep legends with many entries from overflowing the plot
    p.legend.ncols = 2
    p.legend.label_text_font_size = "8pt"
    p.legend.glyph_height = 12
    p.legend.label_height = 12

    return p

# one plot per categorical column, dropdown toggles which one is visible
plots = [make_scatterplot(c) for c in cat_cols]
for p in plots[1:]:
    p.visible = False

dropdown = Select(title="Categorical column", value=cat_cols[0], options=cat_cols)
dropdown.js_on_change("value", CustomJS(args=dict(plots=plots, names=cat_cols), code="""
    plots.forEach((p, i) => p.visible = names[i] === cb_obj.value);
"""))

show(column(dropdown, *plots))


#### Response: Justification for categorical data encoding

Categorical columns are encoded with color. Their values are unordered and categorical, so a categorical palette (size adjusted based on value range) matches well. A legend mapps each hue to a value. To combat overplotting, points are small and semi-transparent. The iso-density contour lines show where points concentrate.

### Assignment V.4 (10 points)

Interactive exploration

In this final assignment we want to do some true interactive exploration, harnessing the power of Bokeh by combining selection and filtering interactions with the charts you already built.
You have two options to proceed, based on your previous results. The below descriptions are minimal solutions, feel free to go beyond the suggested functionality, if you deem it useful. If you do so add a brief list of what you added in the textfield below.

#### Pick one of the the following options:

1. Use the result of **Assignment V.1** and extend it such that clicking a bar in one of the charts affects all the charts. I.e.,
   * Make it possible to select any subset of the data by clicking the corresponding bar in any of the bar/histogram charts. E.g., clicking on the *Bachelors* bar in the *Education* chart should select all datapoints where the value of Education is Bachelors.
   * Render the effect of that selection in every chart (while keeping the full dataset visible too). E.g., render a second bar corresponding to the fraction of the corresponding points within a different color on top of the original bar.
<br/><br/>
2. Combine the results of **Assignment V.2** and **Assignment V.3** into a single linked layout.
   * Show both plots next to each other in a grid.
   * Allow selection of the numerical columns in the scatterplot by clicking on a cell in the heatmap.
   * Reuse or extend the categorical-column dropdown you already built in Assignment V.3.
   * Add a way to select one or multiple values of the categorical column and update the scatterplot based on that selection. E.g., show only the datapoints where *Education* equals *Bachelors*, when Education is selected as the categorical variable of the plot. Ensure that the axis ranges do not change when the user selects values of the categorical column so that plotted points do not shift. 

For your chosen option, explain the analysis task(s) that your interactive visualization allows for. Describe it as one or more **{action, target}** pairs. Say briefly why the interaction you built (rather than a static version of the same charts) is what makes that task possible, **in 50 words or less**.

#### Some possibly useful hints:
* Check out **TapTool** for more information on enabling clicking events on plots
* Have a look at **CDSView** for one possible way to filter out a subset of data on a plot
* [Enabling click callbacks on bars](https://docs.bokeh.org/en/2.4.3/docs/user_guide/tools.html#setting-the-active-tools)
* [Getting clicked bar index in callback functions](https://discourse.bokeh.org/t/a-different-plot-in-another-figure-when-a-bar-of-a-barchart-is-clicked/8572/2)
* [Apply hover tools to plots](https://stackoverflow.com/questions/29435200/bokeh-plotting-enable-tooltips-for-only-some-glyphs)


In [ ]:
# Option 2: link the correlation heatmap (V.2) and categorical scatterplot (V.3).
from bokeh.layouts import row
from bokeh.models import Button, ColorBar, Div, LinearColorMapper, MultiSelect, Range1d
from bokeh.palettes import Category10, Category20, RdBu11, Turbo256
from bokeh.plotting.contour import contour_data


def make_linked_explorer(frame, density_grid_size=40):
    """Build a standalone Bokeh layout; all interactions run in CustomJS."""
    numeric = frame.select_dtypes(include="number")
    categorical = frame.select_dtypes(include=["object", "category", "string", "bool"])
    if numeric.shape[1] < 2 or categorical.shape[1] < 1:
        raise ValueError("At least two numerical columns and one categorical column are required.")

    # Internal keys keep arbitrary input names separate from plotting fields.
    num_keys = [f"n{i}" for i in range(numeric.shape[1])]
    cat_keys = [f"c{i}" for i in range(categorical.shape[1])]
    num_labels = dict(zip(num_keys, map(str, numeric.columns)))
    raw_data, limits, categories = {}, {}, {}
    for key, (_, series) in zip(num_keys, numeric.items()):
        values = series.to_numpy(dtype=float, na_value=np.nan)
        raw_data[key] = values
        finite = values[np.isfinite(values)]
        lo, hi = (float(finite.min()), float(finite.max())) if len(finite) else (0.0, 1.0)
        padding = 0.05 * (hi - lo) if hi > lo else max(abs(lo) * 0.05, 0.5)
        limits[key] = [lo - padding, hi + padding]

    def palette_for(count):
        if count <= 10:
            return list(Category10[10][:count])
        if count <= 20:
            return list(Category20[20][:count])
        # Interpolate the palette so high-cardinality columns have no 256-value limit.
        rgb = np.array([[int(c[j:j + 2], 16) for j in (1, 3, 5)] for c in Turbo256])
        channels = np.array([np.interp(np.linspace(0, 255, count), np.arange(256), rgb[:, j])
                             for j in range(3)]).T.round().astype(int)
        return [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in channels]

    for key, (_, series) in zip(cat_keys, categorical.items()):
        codes, unique = pd.factorize(series, sort=False, use_na_sentinel=False)
        labels = ["(missing)" if pd.isna(value) else str(value) for value in unique]
        ids = [str(i) for i in range(len(labels))]
        raw_data[key] = codes.astype(str)
        categories[key] = dict(ids=ids, labels=labels, colors=palette_for(len(labels)))
    raw = ColumnDataSource(raw_data, name="v4_raw")

    # Keep every matrix cell, including undefined correlations for constant columns.
    clean_numeric = numeric.replace([np.inf, -np.inf], np.nan)
    correlations = {method: clean_numeric.corr(method=method).to_numpy().ravel().tolist()
                    for method in ("pearson", "spearman")}
    x_keys = num_keys * len(num_keys)
    y_keys = [key for key in num_keys for _ in num_keys]
    first_cell = len(num_keys)  # x = first numerical column, y = second
    heat_source = ColumnDataSource(dict(
        x_key=x_keys, y_key=y_keys,
        x_label=[num_labels[k] for k in x_keys], y_label=[num_labels[k] for k in y_keys],
        correlation=correlations["spearman"],
    ), name="v4_heat_source")
    heat_source.selected.indices = [first_cell]
    mapper = LinearColorMapper(palette=RdBu11, low=-1, high=1, nan_color="#d1d5db")
    heat = figure(width=400, height=480, x_range=list(num_labels.values()), y_range=list(reversed(list(num_labels.values()))),
                  title="Spearman correlation · full dataset", tools="tap", toolbar_location=None,
                  name="v4_heatmap")
    heat.rect("x_label", "y_label", width=1, height=1, source=heat_source,
              fill_color={"field": "correlation", "transform": mapper}, line_color="white",
              nonselection_fill_alpha=1, nonselection_line_alpha=1,
              selection_line_color="#111827", selection_line_width=3)
    heat.add_tools(HoverTool(tooltips=[("X", "@x_label"), ("Y", "@y_label"),
                                       ("Correlation", "@correlation{0.000}")]))
    heat.add_layout(ColorBar(color_mapper=mapper, title="r", width=12), "right")
    heat.xaxis.major_label_orientation = np.pi / 3
    heat.xaxis.axis_label = "Click a cell to choose X and Y"
    heat.grid.grid_line_color = None

    # Precompute each unordered pair once, using all finite rows, as in V.3.
    # CustomJS can then swap the contours without a Python server or stale KDE.
    densities = {}
    for i, x_key in enumerate(num_keys):
        for y_key in num_keys[i + 1:]:
            x, y = raw_data[x_key], raw_data[y_key]
            valid = np.isfinite(x) & np.isfinite(y)
            x, y = x[valid], y[valid]
            density = dict(xs=[], ys=[], note="KDE unavailable for this pair (insufficient or singular data).")
            if len(x) >= 3 and np.ptp(x) > 0 and np.ptp(y) > 0:
                try:
                    X, Y = np.mgrid[x.min():x.max():density_grid_size * 1j,
                                    y.min():y.max():density_grid_size * 1j]
                    kernel = gaussian_kde(np.vstack([x, y]))
                    Z = kernel(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
                    levels = np.linspace(Z.min(), Z.max(), 7)[1:-1]
                    lines = contour_data(X, Y, Z, levels=levels, want_fill=False).line_data
                    if lines is not None:
                        density = dict(xs=[list(v) for v in lines.xs], ys=[list(v) for v in lines.ys],
                                       note="Contours: full-data KDE for the current numerical pair.")
                except (np.linalg.LinAlgError, ValueError):
                    pass  # A valid scatterplot need not have an invertible KDE covariance.
            densities[f"{x_key}:{y_key}"] = density

    x_key, y_key, cat_key = num_keys[0], num_keys[1], cat_keys[0]
    meta = categories[cat_key]
    valid = np.isfinite(raw_data[x_key]) & np.isfinite(raw_data[y_key])
    row_ids = np.flatnonzero(valid)
    codes = raw_data[cat_key][valid].astype(int)
    points = ColumnDataSource(dict(
        x=raw_data[x_key][valid], y=raw_data[y_key][valid], row_id=row_ids,
        category=[meta["labels"][i] for i in codes], color=[meta["colors"][i] for i in codes],
    ), name="v4_points")
    scatter = figure(width=530, height=480, title=f"{num_labels[y_key]} vs {num_labels[x_key]}",
                     x_axis_label=num_labels[x_key], y_axis_label=num_labels[y_key],
                     x_range=Range1d(*limits[x_key]), y_range=Range1d(*limits[y_key]),
                     tools="pan,wheel_zoom,box_zoom,reset,save", active_scroll="wheel_zoom",
                     name="v4_scatter")
    dots = scatter.scatter("x", "y", source=points, size=4, alpha=0.35, color="color",
                           line_color=None)
    scatter.add_tools(HoverTool(renderers=[dots], tooltips=[("X", "@x{0,0.###}"),
                          ("Y", "@y{0,0.###}"), ("Category", "@category")]))
    initial_density = densities[f"{x_key}:{y_key}"]
    contour_source = ColumnDataSource(dict(xs=initial_density["xs"], ys=initial_density["ys"]),
                                     name="v4_contours")
    scatter.multi_line("xs", "ys", source=contour_source, line_color="#111827",
                       line_alpha=0.65, line_width=1.2)

    method_select = Select(title="Correlation method", value="spearman",
                           options=[("spearman", "Spearman"), ("pearson", "Pearson")], width=180)
    cat_select = Select(title="Colour / filter by", value=cat_key, width=210,
                        options=list(zip(cat_keys, map(str, categorical.columns))), name="v4_category")
    values_select = MultiSelect(title="Visible values (Cmd/Ctrl-click for multiple)",
                               value=meta["ids"], options=list(zip(meta["ids"], meta["labels"])),
                               size=5, width=400, name="v4_values")
    all_button = Button(label="Select all", width=110)
    none_button = Button(label="Clear selection", width=110)
    count_text = Div(text=f"Showing {len(row_ids):,} / {len(frame):,} rows (finite X and Y only).",
                     render_as_text=True, width=930, name="v4_count")
    density_text = Div(text=initial_density["note"], render_as_text=True, width=930,
                       name="v4_density_note")

    def legend_html(labels, colors):
        def escape(value):
            return value.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;").replace("'", "&#39;")
        return "<b>Colour key</b><br>" + " ".join(
            f'<span style="display:inline-block;margin:3px 14px 3px 0">'
            f'<span style="color:{color}">&#9679;</span> {escape(label)}</span>'
            for label, color in zip(labels, colors))

    legends = {key: legend_html(value["labels"], value["colors"]) for key, value in categories.items()}
    legend = Div(text=legends[cat_key], width=930, height=90, styles={"overflow-y": "auto"},
                 name="v4_legend")
    state = ColumnDataSource(dict(cell=[first_cell], category=[cat_key]))
    update = CustomJS(args=dict(raw=raw, heat_source=heat_source, points=points, scatter=scatter,
        cat_select=cat_select, values_select=values_select, categories=categories, limits=limits,
        num_labels=num_labels, densities=densities, contour_source=contour_source,
        count_text=count_text, density_text=density_text, state=state, legend=legend, legends=legends,
        xaxis=scatter.xaxis[0], yaxis=scatter.yaxis[0]), code="""
        const indices = heat_source.selected.indices;
        const cell = indices.length ? indices[indices.length - 1] : state.data.cell[0];
        const xkey = heat_source.data.x_key[cell], ykey = heat_source.data.y_key[cell];
        const cat = cat_select.value, meta = categories[cat];
        if (cat !== state.data.category[0]) {
            state.data.category[0] = cat;
            values_select.options = meta.ids.map((id, i) => [id, meta.labels[i]]);
            values_select.value = meta.ids.slice();
            legend.text = legends[cat];
        }
        // Range1d is explicit: filtering NEVER modifies these ranges, including after zooming.
        if (cell !== state.data.cell[0]) {
            state.data.cell[0] = cell;
            for (const [range, key] of [[scatter.x_range, xkey], [scatter.y_range, ykey]]) {
                [range.start, range.end] = limits[key];
                [range.reset_start, range.reset_end] = limits[key];
            }
            xaxis.axis_label = num_labels[xkey];
            yaxis.axis_label = num_labels[ykey];
            scatter.title.text = `${num_labels[ykey]} vs ${num_labels[xkey]}`;
            const forward = densities[`${xkey}:${ykey}`];
            const reverse = densities[`${ykey}:${xkey}`];
            const density = forward || reverse;
            contour_source.data = density
                ? {xs: forward ? density.xs : density.ys, ys: forward ? density.ys : density.xs}
                : {xs: [], ys: []};
            density_text.text = density ? density.note : "Same variable on both axes: no 2D KDE.";
        }
        const selected = new Set(values_select.value);
        const xs = raw.data[xkey], ys = raw.data[ykey], cats = raw.data[cat];
        const next = {x: [], y: [], row_id: [], category: [], color: []};
        for (let i = 0; i < xs.length; i++) {
            if (Number.isFinite(xs[i]) && Number.isFinite(ys[i]) && selected.has(cats[i])) {
                const code = Number(cats[i]);
                next.x.push(xs[i]); next.y.push(ys[i]); next.row_id.push(i);
                next.category.push(meta.labels[code]); next.color.push(meta.colors[code]);
            }
        }
        points.data = next;
        count_text.text = `Showing ${next.x.length.toLocaleString()} / ${xs.length.toLocaleString()} rows (finite X and Y only).`;
    """)
    heat_source.selected.js_on_change("indices", update)
    cat_select.js_on_change("value", update)
    values_select.js_on_change("value", update)
    method_select.js_on_change("value", CustomJS(args=dict(source=heat_source, heat=heat,
        correlations=correlations), code="""
        source.data = {...source.data, correlation: correlations[cb_obj.value]};
        heat.title.text = `${cb_obj.value === "pearson" ? "Pearson" : "Spearman"} correlation · full dataset`;
    """))
    all_button.js_on_click(CustomJS(args=dict(select=values_select, cat=cat_select, categories=categories),
        code="select.value = categories[cat.value].ids.slice();"))
    none_button.js_on_click(CustomJS(args=dict(select=values_select), code="select.value = [];"))
    return column(
        Div(text="<h3>Linked correlation and category explorer</h3>"
                 "Click a heatmap cell to change X/Y. Choose a categorical column and select values to compare groups. "
                 "Grey cells have undefined correlations. Heatmap and contours describe the full dataset.", width=930),
        row(method_select, cat_select, values_select, column(all_button, none_button)),
        count_text, gridplot([[heat, scatter]], merge_tools=False), density_text, legend,
    )


part4_layout = make_linked_explorer(data)
show(part4_layout)


#### Response: Task(s) your visualization supports

{Explore, numerical relationships} by clicking correlation cells; {compare, category-specific distributions} by selecting groups. Linked views let users move from correlation summaries to individual observations. Fixed axes preserve spatial context during filtering, making group differences easier to compare than separate static plots.

**Additional functionality:** Pearson/Spearman switching, select-all/clear buttons, a live row count, hover values, pan/zoom/reset, and full-data KDE contours that update with the selected numerical pair. Undefined correlations appear grey; singular or same-variable pairs omit KDE. Category changes reset the value filter to all groups.


#### AI Use

YOUR RESPONSE HERE